# Fine-Tuning End-to-End Test

End-to-end test of the household fine-tuning pipeline against a **realistic synthetic Lebanese household** ground truth.

**Two-week design:**
- **Week 1 (Jul 1–7):** generate synthetic ground truth, fine-tune base models
- **Week 2 (Jul 8–14):** generate a fresh ground truth, evaluate base and fine-tuned models on held-out data

**Graphs produced (all evaluated on week 2):**
- **A** — Weather during week 2
- **B** — Week 2 ground truth (held-out)
- **C** — Base model predictions on week 2
- **D** — Fine-tuned model predictions on week 2
- **E** — F1 score comparison on week 2
- **F/G** — Cooling feature importance (base vs fine-tuned)

**Output:** `test_finetuning_results.png`

## Imports & Configuration

In [20]:
import os
import sys
import pickle
import warnings
import tempfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
from tuning import run_finetuning

warnings.filterwarnings("ignore")

BASE_MODELS_DIR = r"C:\Users\moham\Documents\490 project new\final_models"
HOUSEHOLDS_DIR  = r"C:\Users\moham\Documents\490 project new\household_models"
OUTPUT_PATH     = r"C:\Users\moham\Documents\490 project new\test_finetuning_results.png"

LAT, LON, TZ    = 33.89, 35.50, "Asia/Beirut"
TRAIN_DATE_START = "2024-07-01"
TRAIN_DATE_END   = "2024-07-07"  # week 1 — fine-tuning data
TEST_DATE_START  = "2024-07-08"
TEST_DATE_END    = "2024-07-14"  # week 2 — held-out evaluation

APPLIANCES = [
    "elec_cooling_on",
    "elec_clothes_washer_on",
    "elec_hot_water_on",
    "elec_television_on",
    "elec_heating_on",
]

ALWAYS_ON = ["fridge", "freezer"]

APPLIANCE_LABELS = {
    "elec_cooling_on":         "Cooling (AC)",
    "elec_clothes_washer_on":  "Washing Machine",
    "elec_hot_water_on":       "Hot Water",
    "elec_television_on":      "Television",
    "elec_heating_on":         "Heating",
    "fridge":                  "Fridge",
    "freezer":                 "Freezer",
}

FEATURE_COLS = [
    "weather_drybulb_temp_c",
    "weather_relative_humidity_pct",
    "hour",
    "day_of_week",
    "is_weekend",
    "month",
]

# TV was trained without weather and with is_evening instead of month
TV_FEATURE_COLS = [
    "hour",
    "day_of_week",
    "is_weekend",
    "is_evening",
]

## Lebanese Household Usage Patterns

Realistic per-hour probability weights for each appliance, calibrated for a Lebanese summer household (Beirut, July).

In [12]:
def get_features_for(appliance, df, model=None):
    """Returns the right feature DataFrame for a given appliance.
    If a model is provided, uses its stored feature_names_in_ to
    guarantee the column order matches exactly what it was trained on.
    Falls back to default lists if the model has no stored names."""
    if model is not None and hasattr(model, "feature_names_in_"):
        cols = [str(c) for c in model.feature_names_in_]
        return df[cols]
    if appliance == "elec_television_on":
        return df[TV_FEATURE_COLS]
    return df[FEATURE_COLS]

# Lebanese household usage patterns — realistic probabilities
# per hour for each appliance
LEBANESE_PATTERNS = {
    # Cooling: heavy in summer afternoons/evenings
    "elec_cooling_on": {
        "hour_weights": {
            0:0.3,1:0.2,2:0.1,3:0.1,4:0.1,5:0.1,
            6:0.1,7:0.2,8:0.3,9:0.3,10:0.4,11:0.5,
            12:0.7,13:0.9,14:1.0,15:1.0,16:1.0,17:0.9,
            18:0.9,19:0.9,20:0.8,21:0.7,22:0.5,23:0.4,
        },
        "temp_boost": True,   # more likely when hot
    },
    # Washing machine: 1-2 cycles per week, weekday mornings
    "elec_clothes_washer_on": {
        "hour_weights": {
            9:0.4,10:0.6,11:0.5,
        },
        "days": [0, 2, 4],    # Mon, Wed, Fri only
        "max_days": 2,        # max 2 cycles per week
        "duration": 2,        # runs for 2 consecutive hours
    },
    # Hot water: morning showers + evening
    "elec_hot_water_on": {
        "hour_weights": {
            6:0.7,7:0.9,8:0.8,9:0.4,
            20:0.5,21:0.7,22:0.6,
        },
    },
    # Television: evenings, more on weekends
    "elec_television_on": {
        "hour_weights": {
            7:0.1,8:0.1,
            12:0.3,13:0.2,
            18:0.4,19:0.7,20:0.9,21:1.0,22:0.9,23:0.6,
        },
        "weekend_boost": 1.4,
    },
    # Heating: not used in July (summer)
    "elec_heating_on": {
        "hour_weights": {},   # empty = never on in July
    },
}

## Weather Fetch & Mock Fallback

In [13]:
def fetch_weather(start, end):
    print("Fetching weather from Open-Meteo...")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   LAT, "longitude": LON,
        "start_date": start, "end_date": end,
        "hourly":     "temperature_2m,relative_humidity_2m",
        "timezone":   TZ,
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame({
        "timestamp":                     pd.to_datetime(data["hourly"]["time"]),
        "weather_drybulb_temp_c":        data["hourly"]["temperature_2m"],
        "weather_relative_humidity_pct": data["hourly"]["relative_humidity_2m"],
    })
    df["hour"]        = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)
    df["month"]       = df["timestamp"].dt.month
    df["is_evening"]  = df["hour"].between(18, 23).astype(int)
    print(f"  Got {len(df)} hourly rows, "
          f"temp {df['weather_drybulb_temp_c'].min():.1f}–"
          f"{df['weather_drybulb_temp_c'].max():.1f}°C")
    return df


def mock_weather(start, end):
    """Fallback if Open-Meteo is unavailable."""
    print("  [!] Using mock weather (Open-Meteo unavailable)")
    times = pd.date_range(start, periods=336, freq="h")
    hours = np.array([t.hour for t in times])
    # Realistic Beirut July temperatures: 26-34°C
    temp  = 30 + 4 * np.sin((hours - 14) * np.pi / 12) + np.random.normal(0, 0.5, len(times))
    humid = 65 - 10 * np.sin((hours - 14) * np.pi / 12) + np.random.normal(0, 2, len(times))
    df = pd.DataFrame({
        "timestamp":                     times,
        "weather_drybulb_temp_c":        np.clip(temp, 22, 38),
        "weather_relative_humidity_pct": np.clip(humid, 40, 85),
    })
    df["hour"]        = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)
    df["month"]       = df["timestamp"].dt.month
    df["is_evening"]  = df["hour"].between(18, 23).astype(int)
    return df


## Base Model Loader

In [14]:
def load_base_models(base_dir):
    """
    Loads all base .pkl models found in base_dir.
    Returns dict: appliance_name -> XGBClassifier
    Also creates placeholder models for appliances with no .pkl
    so the test can always run end-to-end.
    """
    models = {}
    for app in APPLIANCES:
        path = os.path.join(base_dir, f"{app}.pkl")
        if os.path.exists(path):
            with open(path, "rb") as f:
                models[app] = pickle.load(f)
            print(f"  Loaded: {app}.pkl")
        else:
            print(f"  [!] No base model for {app} — creating placeholder")
            # Placeholder trained on random data so the script runs
            X_p = np.random.rand(300, 6)
            y_p = (np.random.rand(300) > 0.7).astype(int)
            m   = XGBClassifier(n_estimators=50, random_state=42)
            m.fit(X_p, y_p)
            models[app] = m
    return models

## Schedule Generator

In [15]:
def generate_schedule(df_weather, models):
    """
    Runs each base model on the weather/time features to produce
    a 7-day hourly binary schedule.
    Returns dict: appliance -> array[168]
    """
    schedule = {}
    for app, model in models.items():
        X = get_features_for(app, df_weather, model)
        preds = model.predict(X)
        schedule[app] = preds.astype(int)
    for app in ALWAYS_ON:
        schedule[app] = np.ones(len(df_weather), dtype=int)
    return schedule

## Synthetic Lebanese Ground Truth Generator

In [16]:
def generate_lebanese_truth(df_weather):
    """
    Generates a realistic Lebanese household schedule for the test week.
    Uses the LEBANESE_PATTERNS dict above with stochastic sampling.
    """
    np.random.seed(99)
    n = len(df_weather)
    schedule = {}

    for app in APPLIANCES:
        pattern = LEBANESE_PATTERNS[app]
        hour_w  = pattern.get("hour_weights", {})
        arr     = np.zeros(n, dtype=int)

        if not hour_w:
            schedule[app] = arr
            continue

        if "duration" in pattern:
            # Block appliances: choose specific days and run for N hours
            allowed_days = pattern.get("days", list(range(7)))
            max_days     = pattern.get("max_days", len(allowed_days))
            duration     = pattern["duration"]
            chosen_days  = np.random.choice(allowed_days,
                                             size=min(max_days, len(allowed_days)),
                                             replace=False)
            for day_idx in range(7):
                if day_idx not in chosen_days:
                    continue
                # Pick start hour weighted by hour_weights
                hours_avail = list(hour_w.keys())
                weights_arr = np.array([hour_w[h] for h in hours_avail])
                weights_arr = weights_arr / weights_arr.sum()
                start_h = np.random.choice(hours_avail, p=weights_arr)
                for d in range(duration):
                    idx = day_idx * 24 + start_h + d
                    if idx < n:
                        arr[idx] = 1
        else:
            # Probabilistic hourly appliances
            weekend_boost = pattern.get("weekend_boost", 1.0)
            for i in range(n):
                hour    = df_weather["hour"].iloc[i]
                is_wknd = df_weather["is_weekend"].iloc[i]
                prob    = hour_w.get(hour, 0.0)

                if pattern.get("temp_boost") and prob > 0:
                    temp  = df_weather["weather_drybulb_temp_c"].iloc[i]
                    # Boost probability when temperature > 28°C
                    prob  = min(prob * (1 + max(0, (temp - 28) / 10)), 1.0)

                if is_wknd:
                    prob = min(prob * weekend_boost, 1.0)

                arr[i] = 1 if np.random.rand() < prob else 0

        schedule[app] = arr

    for app in ALWAYS_ON:
        schedule[app] = np.ones(n, dtype=int)

    return schedule

## Fine-Tuning (delegates to production pipeline)

In [17]:
def finetune_models(df_weather, truth_schedule, base_models=None):
    """
    Delegates to the production run_finetuning() pipeline.
    Saves ground truth to a temp CSV, runs the pipeline,
    loads and returns the saved fine-tuned models.
    Fine-tuning settings live in finetune_household.py only.
    """
    tmp_csv = tempfile.mktemp(suffix=".csv")
    truth_df = pd.DataFrame(
        {"timestamp": df_weather["timestamp"],
         **{a: truth_schedule[a] for a in APPLIANCES}}
    )
    truth_df.to_csv(tmp_csv, index=False)

    run_finetuning(
        household_id="test_household",
        usage_csv_path=tmp_csv,
        lat=LAT, lon=LON, tz=TZ,
        base_models_dir=BASE_MODELS_DIR,
        households_dir=HOUSEHOLDS_DIR,
        prefetched_weather=df_weather,
    )
    os.remove(tmp_csv)

    ft_models = {}
    hh_dir = os.path.join(HOUSEHOLDS_DIR, "test_household")
    for app in APPLIANCES:
        path = os.path.join(hh_dir, f"{app}.pkl")
        if os.path.exists(path):
            with open(path, "rb") as f:
                ft_models[app] = pickle.load(f)
            print(f"  Loaded fine-tuned: {app}")
        else:
            print(f"  [!] No fine-tuned model for {app} — using base")
            with open(os.path.join(BASE_MODELS_DIR, f"{app}.pkl"), "rb") as _f:
                ft_models[app] = pickle.load(_f)
    return ft_models

## Plotting Helpers

In [18]:
CMAP_BIN = ListedColormap(["#f0f0f0", "#2196F3"])
DAY_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

def schedule_to_matrix(schedule, apps):
    """Converts schedule dict to (n_apps x 7days x 24hours) matrix."""
    rows = []
    for app in apps:
        arr = np.array(schedule[app][:168]).reshape(7, 24)
        rows.append(arr)
    return np.array(rows)   # shape: (n_apps, 7, 24)


def plot_schedule_heatmap(ax, matrix, app_labels, title, show_hours=False):
    """Plots a 7-day schedule as a heatmap: rows=days, cols=hours."""
    n_apps = len(app_labels)
    # Stack days horizontally: shape (n_apps, 168)
    flat = matrix.reshape(n_apps, 168)

    ax.imshow(flat, aspect="auto", cmap=CMAP_BIN, vmin=0, vmax=1,
              interpolation="nearest")
    ax.set_yticks(range(n_apps))
    ax.set_yticklabels(app_labels, fontsize=8)
    ax.set_title(title, fontsize=10, fontweight="bold", pad=4)

    # Day separators
    for d in range(1, 7):
        ax.axvline(d * 24 - 0.5, color="white", linewidth=1.5)

    # Day labels on x-axis
    ax.set_xticks([d * 24 + 11.5 for d in range(7)])
    ax.set_xticklabels(DAY_LABELS, fontsize=8)
    ax.tick_params(axis="x", length=0)

    if show_hours:
        ax.set_xlabel("Hour of day →", fontsize=8)


def plot_on_hours_comparison(ax, schedules, labels, colors, app_labels):
    """Bar chart comparing total ON hours per appliance across schedules."""
    x    = np.arange(len(app_labels))
    w    = 0.8 / len(schedules)
    for i, (sched, label, color) in enumerate(zip(schedules, labels, colors)):
        on_hrs = [sched[a].sum() for a in APPLIANCES]
        ax.bar(x + i * w, on_hrs, width=w, label=label,
               color=color, alpha=0.85, edgecolor="white", linewidth=0.5)
    ax.set_xticks(x + w)
    ax.set_xticklabels(app_labels, fontsize=8, rotation=20, ha="right")
    ax.set_ylabel("Total ON hours (7 days)", fontsize=8)
    ax.set_title("ON hours comparison", fontsize=10, fontweight="bold", pad=4)
    ax.legend(fontsize=8, framealpha=0.7)
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.spines[["top","right"]].set_visible(False)


def plot_feature_importance(ax, model, title):
    """Horizontal bar chart of feature importances by gain."""
    try:
        scores = model.get_booster().get_score(importance_type="gain")
    except Exception:
        ax.set_visible(False)
        return
    if not scores:
        ax.set_visible(False)
        return
    feat_labels = {
        "f0": "Temperature",
        "f1": "Humidity",
        "f2": "Hour",
        "f3": "Day of week",
        "f4": "Is weekend",
        "f5": "Month",
    }
    items  = sorted(scores.items(), key=lambda x: x[1])
    keys   = [feat_labels.get(k, k) for k, _ in items]
    values = [v for _, v in items]
    bars   = ax.barh(keys, values, color="#42A5F5", edgecolor="white", linewidth=0.5)
    ax.set_title(title, fontsize=10, fontweight="bold", pad=4)
    ax.set_xlabel("Gain", fontsize=8)
    ax.tick_params(labelsize=8)
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3, linewidth=0.5)

## Main — Run Full Test & Produce Figure

In [19]:
def main():
    print("\n" + "="*60)
    print("FINE-TUNING TEST — Lebanese Household (2-Week)")
    print("="*60)

    # ── Fetch weather for both weeks ────────────────────────────────
    print("\nFetching weather for week 1 (fine-tuning)...")
    try:
        df_train = fetch_weather(TRAIN_DATE_START, TRAIN_DATE_END)
    except Exception as e:
        print(f"  Open-Meteo failed ({e}), using mock weather")
        df_train = mock_weather(TRAIN_DATE_START, TRAIN_DATE_END)
    df_train = df_train.head(168).reset_index(drop=True)

    print("Fetching weather for week 2 (evaluation)...")
    try:
        df_test = fetch_weather(TEST_DATE_START, TEST_DATE_END)
    except Exception as e:
        print(f"  Open-Meteo failed ({e}), using mock weather")
        df_test = mock_weather(TEST_DATE_START, TEST_DATE_END)
    df_test = df_test.head(168).reset_index(drop=True)

    # ── Base models ───────────────────────────────────────────
    print("\nLoading base models...")
    base_models = load_base_models(BASE_MODELS_DIR)

    # ── Week 1: ground truth + fine-tuning ──────────────────────
    print("\nGenerating week 1 Lebanese ground truth (fine-tuning data)...")
    train_truth = generate_lebanese_truth(df_train)

    print("\nFine-tuning models on week 1 data...")
    ft_models = finetune_models(df_train, train_truth)

    # ── Week 2: generate schedules + ground truth for evaluation ────
    print("\nGenerating week 2 Lebanese ground truth (held-out evaluation)...")
    test_truth = generate_lebanese_truth(df_test)

    print("\nGenerating base model schedule on week 2...")
    base_sched = generate_schedule(df_test, base_models)

    print("\nGenerating fine-tuned schedule on week 2...")
    ft_sched = generate_schedule(df_test, ft_models)

    # ── Build matrices for plotting (all use week 2) ──────────────
    all_apps   = APPLIANCES + ALWAYS_ON
    app_labels = [APPLIANCE_LABELS[a] for a in APPLIANCES]
    all_labels = [APPLIANCE_LABELS[a] for a in all_apps]

    base_mat  = schedule_to_matrix(base_sched,  all_apps)
    truth_mat = schedule_to_matrix(test_truth,  all_apps)
    ft_mat    = schedule_to_matrix(ft_sched,    all_apps)

    # ── Figure layout ─────────────────────────────────────────
    fig = plt.figure(figsize=(20, 22), facecolor="white")
    gs  = gridspec.GridSpec(
        5, 2,
        figure=fig,
        hspace=0.45,
        wspace=0.35,
        left=0.08, right=0.97,
        top=0.95,  bottom=0.05,
    )

    # ── A: Week 2 weather ───────────────────────────────────────
    ax_temp = fig.add_subplot(gs[0, 0])
    ax_hum  = ax_temp.twinx()
    hours   = range(168)

    ax_temp.plot(hours, df_test["weather_drybulb_temp_c"],
                 color="#E53935", linewidth=1.2, label="Temperature (°C)")
    ax_hum.plot(hours, df_test["weather_relative_humidity_pct"],
                color="#1E88E5", linewidth=1.0, alpha=0.7,
                linestyle="--", label="Humidity (%)")

    for d in range(1, 7):
        ax_temp.axvline(d * 24, color="gray", linewidth=0.5, alpha=0.4)

    ax_temp.set_xticks([d * 24 + 11.5 for d in range(7)])
    ax_temp.set_xticklabels(DAY_LABELS, fontsize=8)
    ax_temp.set_ylabel("Temperature (°C)", fontsize=8, color="#E53935")
    ax_hum.set_ylabel("Humidity (%)", fontsize=8, color="#1E88E5")
    ax_temp.set_title("A — Beirut weather, week 2 (8–14 Jul 2024)", fontsize=10,
                      fontweight="bold", pad=4)
    lines1, labs1 = ax_temp.get_legend_handles_labels()
    lines2, labs2 = ax_hum.get_legend_handles_labels()
    ax_temp.legend(lines1 + lines2, labs1 + labs2, fontsize=7,
                   loc="upper right", framealpha=0.7)
    ax_temp.tick_params(axis="y", colors="#E53935", labelsize=8)
    ax_hum.tick_params(axis="y", colors="#1E88E5", labelsize=8)
    ax_temp.tick_params(axis="x", labelsize=8, length=0)
    ax_temp.spines[["top"]].set_visible(False)

    # ── B: Week 2 ground truth ─────────────────────────────────
    ax_truth = fig.add_subplot(gs[0, 1])
    plot_schedule_heatmap(ax_truth, truth_mat, all_labels,
                          "B — Week 2 ground truth (held-out)")

    # ── C: Base model on week 2 ───────────────────────────────
    ax_base = fig.add_subplot(gs[1, 0])
    plot_schedule_heatmap(ax_base, base_mat, all_labels,
                          "C — Base model on week 2 (before fine-tuning)")

    # ── D: Fine-tuned model on week 2 ──────────────────────────
    ax_ft = fig.add_subplot(gs[1, 1])
    plot_schedule_heatmap(ax_ft, ft_mat, all_labels,
                          "D — Fine-tuned model on week 2 (after fine-tuning)",
                          show_hours=True)

    # ── E: ON hours comparison (week 2) ─────────────────────────
    ax_bar = fig.add_subplot(gs[2, :])
    plot_on_hours_comparison(
        ax_bar,
        schedules=[test_truth, base_sched, ft_sched],
        labels=["Ground truth (wk2)", "Base model (wk2)", "Fine-tuned (wk2)"],
        colors=["#43A047", "#FB8C00", "#1E88E5"],
        app_labels=app_labels,
    )

    # ── F: F1 scores on week 2 ─────────────────────────────────
    ax_f1 = fig.add_subplot(gs[3, :])
    f1_base = []
    f1_ft   = []
    for app in APPLIANCES:
        y_true = test_truth[app]
        X_app  = get_features_for(app, df_test, base_models[app])
        if y_true.sum() == 0:
            f1_base.append(0.0)
            f1_ft.append(0.0)
            continue
        f1_base.append(f1_score(y_true, base_models[app].predict(X_app),
                                zero_division=0))
        f1_ft.append(f1_score(y_true, ft_models[app].predict(X_app),
                              zero_division=0))

    x_pos = np.arange(len(APPLIANCES))
    w     = 0.35
    ax_f1.bar(x_pos - w/2, f1_base, w, label="Base model",
              color="#FB8C00", alpha=0.85, edgecolor="white")
    ax_f1.bar(x_pos + w/2, f1_ft,   w, label="Fine-tuned",
              color="#1E88E5", alpha=0.85, edgecolor="white")
    ax_f1.set_xticks(x_pos)
    ax_f1.set_xticklabels(app_labels, fontsize=8, rotation=20, ha="right")
    ax_f1.set_ylabel("F1 score vs week 2 ground truth", fontsize=8)
    ax_f1.set_ylim(0, 1.1)
    ax_f1.set_title("E — F1 score on held-out week 2 (base vs fine-tuned)",
                    fontsize=10, fontweight="bold", pad=4)
    ax_f1.legend(fontsize=8, framealpha=0.7)
    ax_f1.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax_f1.spines[["top","right"]].set_visible(False)
    for i, (b, f) in enumerate(zip(f1_base, f1_ft)):
        ax_f1.text(i - w/2, b + 0.02, f"{b:.2f}", ha="center",
                   fontsize=7, color="#FB8C00", fontweight="bold")
        ax_f1.text(i + w/2, f + 0.02, f"{f:.2f}", ha="center",
                   fontsize=7, color="#1E88E5", fontweight="bold")

    # ── G: Feature importance — cooling ──────────────────────
    ax_imp_base = fig.add_subplot(gs[4, 0])
    ax_imp_ft   = fig.add_subplot(gs[4, 1])
    plot_feature_importance(ax_imp_base, base_models["elec_cooling_on"],
                            "F — Cooling feature importance (base model)")
    plot_feature_importance(ax_imp_ft,   ft_models["elec_cooling_on"],
                            "G — Cooling feature importance (fine-tuned)")

    # ── Title ─────────────────────────────────────────────────
    fig.suptitle(
        "Household Fine-Tuning Test — Beirut, July 2024\n"
        "Fine-tuned on Jul 1–7 • Evaluated on Jul 8–14",
        fontsize=13, fontweight="bold", y=0.975,
    )

    # ── Colorbar legend for heatmaps ──────────────────────────
    from matplotlib.patches import Patch
    legend_elems = [
        Patch(facecolor="#f0f0f0", edgecolor="gray", label="OFF"),
        Patch(facecolor="#2196F3", edgecolor="gray", label="ON"),
    ]
    fig.legend(handles=legend_elems, loc="upper right",
               bbox_to_anchor=(0.98, 0.97), fontsize=8,
               title="Schedule", title_fontsize=8, framealpha=0.8)

    plt.savefig(OUTPUT_PATH, dpi=150, bbox_inches="tight",
                facecolor="white")
    print(f"\nFigure saved to: {OUTPUT_PATH}")
    return OUTPUT_PATH


main()


FINE-TUNING TEST — Lebanese Household (2-Week)

Fetching weather for week 1 (fine-tuning)...
Fetching weather from Open-Meteo...
  Got 168 hourly rows, temp 24.1–32.1°C
Fetching weather for week 2 (evaluation)...
Fetching weather from Open-Meteo...
  Got 168 hourly rows, temp 25.7–33.2°C

Loading base models...
  Loaded: elec_cooling_on.pkl
  Loaded: elec_clothes_washer_on.pkl
  Loaded: elec_hot_water_on.pkl
  Loaded: elec_television_on.pkl
  Loaded: elec_heating_on.pkl

Generating week 1 Lebanese ground truth (fine-tuning data)...

Fine-tuning models on week 1 data...

FINE-TUNING HOUSEHOLD: test_household
  Loaded 168 hourly rows
  Date range: 2024-07-01 00:00:00 → 2024-07-07 23:00:00
  Fetching weather for 2024-07-01 → 2024-07-07 ...
  Using pre-fetched weather (168 rows)
  Final rows after weather merge: 168

  Detected appliances: ['elec_cooling_on', 'elec_clothes_washer_on', 'elec_hot_water_on', 'elec_television_on', 'elec_heating_on']

  Appliance: elec_cooling_on
    ON: 86 (5

'C:\\Users\\moham\\Documents\\490 project new\\test_finetuning_results.png'